In [2]:
import py_dss_interface
from py_dss_toolkit import dss_tools

dss = py_dss_interface.DSS()

dss_file = r"C:\Dados_teste\OpenDSS\CBA\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"
#dss_file = r"C:\Users\IVAN SANTOS\Documents\OpenDSS\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"

dss_tools.update_dss(dss)

dss.text(f"compile [{dss_file}]")

#dss.text("Redirect 'PV_System_60_MeiaPonte.dss'")

dss.text(f"buscoords BusCoords.csv")

dss.text("set mode=daily")
dss.text("set stepsize=1h")
dss.text("set number=24")
dss.solution.solve()

df_energymeter = dss_tools.results.energymeters

display(df_energymeter)

,name,kwh,kvarh,max kw,max kva,zone kwh,zone kvarh,zone max kw,zone max kva,overload kwh normal,...,aux19,aux24,aux29,13.8 kv load energy,aux5,aux10,aux15,aux20,aux25,aux30
0,busa4-_2021126072_segmmt_53951427,124421.081474,63076.686747,7065.687191,8103.461649,118141.936302,49529.463882,6612.56024,7223.772169,107.853497,...,0.0,0.0,0.0,118141.936302,0.0,0.0,0.0,0.0,0.0,0.0


Cargas

In [3]:
loads_df = dss_tools.model.loads_df
loads_df

,name,phases,bus1,kv,kw,pf,model,yearly,daily,duty,...,zipv,%seriesrl,relweight,vlowpu,puxharm,xrharm,spectrum,basefreq,enabled,like
0,mt_10009727955_m1,3,88185385002086.1.2.3.4,13.8,5.46925728729341,0.9021,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
1,mt_10009727955_m2,3,88185385002086.1.2.3.4,13.8,5.46925728729341,0.9021,3,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
2,mt_10014416466_m1,3,89458365002086.1.2.3.4,13.8,2.65068009244008,0.9021,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
3,mt_10014416466_m2,3,89458365002086.1.2.3.4,13.8,2.65068009244008,0.9021,3,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
4,mt_10015559007_m1,3,89813615002086.1.2.3.4,13.8,2.32027223167495,0.9021,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,m1_trf_5333902a,3,85167275333902.1.2.3.4,0.38,26.2254249405443,0.9076,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
302,m1_trf_5740255a,3,82056945740255.1.2.3.4,0.38,14.5905864350407,0.9101,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
303,m1_trf_5740270a,3,96033635740270.1.2.3.4,0.38,12.2445576749837,0.9097,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,
304,m1_trf_5744153a,3,96264655744153.1.2.3.4,0.38,16.4320475658323,0.9072,2,,CURVA,,...,,50,1,0.5,0,6,defaultload,60,true,


Mapas

In [4]:
dss_tools.results.set_violation_voltage_ln_limits( v_min_pu=0.92, v_max_pu=1.05)
dss_tools.interactive_view.circuit_geoplot("voltage violations", show=True, title="Mapa - Violações de Tensão")

In [8]:
import os
print(os.getcwd())
dss_tools.interactive_view.circuit_geoplot(
    parameter="active power", 
    show=True, 
    save_file_path="circuito.html",
    width_3ph=1,
    width_2ph=1,
    width_1ph=1)

C:\Dados_teste\OpenDSS\CBA\Alim_Meia_Ponte_5_REDUZIDO


Mapa Violações Térmicas

In [5]:
import py_dss_interface
from py_dss_toolkit import dss_tools
import pandas as pd
import pickle

dss = py_dss_interface.DSS()

dss_file = r"C:\Dados_teste\OpenDSS\CBA\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"
#dss_file = r"C:\Users\IVAN SANTOS\Documents\OpenDSS\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"

dss_tools.update_dss(dss)

dss.text(f"compile [{dss_file}]")

cases = ["base", "60_gd", "60_gd_30_bess", "120_gd", "120_gd_60_bess"]
titles = {
    "base": "Caso Base",
    "60_gd": "Caso com 60 GDs",
    "60_gd_30_bess": "Caso com 60 GDs e 30 Baterias",
    "120_gd": "Caso com 120 GDs",
    "120_gd_60_bess": "Caso com 120 GDs e 60 Baterias",
}


dfs_monitor = {}
df_energymeter = pd.DataFrame(columns=["Caso", "Perdas (kWh)", "Energia (kWh)", "Energia injetada (kWh)"])
dfs_voltage = {}
dfs_monitor_pf = {}
dfs_storage = {}
dfs_monitor_pv = {}
dfs_battery = {}

for case in cases:

    dss.text(f"compile [{dss_file}]")

    if case == "60_gd":
        dss.text("Redirect 'PV_System_60_MeiaPonte.dss'")
        dss.text("Redirect 'Monitor_PV.dss'")
    
    elif case == "120_gd":
        dss.text("Redirect 'PV_System_120_MeiaPonte.dss'")
        dss.text("Redirect 'Monitores_PV_120.dss'")
    
    elif case == "60_gd_30_bess":
        dss.text("Redirect 'PV_System_60_MeiaPonte.dss'")
        #dss.text("Redirect 'BATERIAS_30.dss'")
        dss.text("Redirect 'Battery_30.dss'")
        dss.text("Redirect 'Monitor_Baterias_30.dss'")

    elif case == "120_gd_60_bess":
        dss.text("Redirect 'PV_System_120_MeiaPonte.dss'")
        #dss.text("Redirect 'BATERIAS_60.dss'")
        dss.text("Redirect 'Battery_60.dss'")
        dss.text("Redirect 'Monitor_Baterias_60.dss'")

    else:
        dss.text("!Redirect 'BATERIAS_60.dss'")
    
    dss.text(f"buscoords BusCoords.csv")

    dss.text("set mode=daily")
    dss.text("set stepsize=1h")
    dss.text("set number=1")

    socs = []

    for h in range(24):
        #if dss.solution.hour == 24:
            #dss.meters.reset_all()
        dss.solution.solve()

        if dss.storages.count > 0:
            dss.storages.first()
            dss.storages.next()   #Para pegar a bateria 2, comentar se for pegar a 1
            name_battery = dss.storages.name
            soc = dss.storages.pu_soc
            socs.append(soc)

    #dss.text("sample")

    dss_tools.results.set_violation_current_limit_type(limit_type="emerg_amps")
    dss_tools.results.set_currents_loading_threshold_percent(threshold_percent=20)
    dss_tools.interactive_view.circuit_plot(
        parameter="thermal violations", 
        title=f"Violações Térmicas (20% de carregamento) - {titles[case]}", 
        show=True)

    if dss.storages.count > 0:
        dfs_storage[f"{name_battery}_{case}"] = socs

    meter_values = dss.meters.register_values

    #print(f"\n {titles[case]}\n")
    #print(f"Energia total: {meter_values[0]:.2f} kWh")
    #print(f"Perdas totais: {meter_values[12]:.2f} kWh\n")

    ##### Monitor - Power Flow Substation ####

    dfs_monitor[f"{case}_df"] = dss_tools.results.monitor("feeder")
    dfs_monitor[f"{case}_df"]["Total Power"] = dfs_monitor[f"{case}_df"][' P1 (kW)'] + dfs_monitor[f"{case}_df"][' P2 (kW)'] + dfs_monitor[f"{case}_df"][' P3 (kW)']

    ##### Monitors - Voltage #####

    monitor_names = dss.monitors.names

    for monitor_name in monitor_names:
        if "voltage" in monitor_name.lower():
            dfs_voltage[f"{monitor_name}_{case}_df"] = dss_tools.results.monitor(monitor_name)
        
        elif "battery" in monitor_name.lower():
            dfs_battery[f"{monitor_name}_{case}_df"] = dss_tools.results.monitor(monitor_name)
        ##### Monitors - Power Flow #####

        else:
            dfs_monitor_pf[f"{monitor_name}_{case}_df"] = dss_tools.results.monitor(monitor_name)

    
    injection = 0
    for chave, df in dfs_monitor_pf.items():    #Items() retorna a tupla (chave, item)

        if chave.endswith(f"_{case}_df") and "trf_" in chave:
            if " P3 (kW)" in df.columns:
                df["Total Power"] = df[" P1 (kW)"] + df[" P2 (kW)"] + df[" P3 (kW)"]
            else:
                df["Total Power"] = df[" P1 (kW)"] + df[" P2 (kW)"]

            #injection += df.query("`Total Power` < 0 and Hour > 24")["Total Power"].sum()  #Descomentar para 48h
            injection += df.query("`Total Power` < 0")["Total Power"].sum()   #Comentar para 48h


    ### Monitors - Geração PV

    for chave in dfs_monitor_pf:
        if chave.endswith(f"_{case}_df") and "pv_" in chave:
            dfs_monitor_pv[chave] = dfs_monitor_pf[chave]

    with open("dataframes.pkl", "wb") as f:
        pickle.dump(dfs_monitor_pv, f)

    ######## Criação do DataFrame "Energymeter" #####
    nova_linha = pd.DataFrame([{
        "Caso": titles[case], 
        "Energia (kWh)": meter_values[0], 
        "Perdas (kWh)": meter_values[12],
        "Energia injetada (kWh)": injection
    }])

    df_energymeter = pd.concat([df_energymeter, nova_linha], ignore_index=True)

#display(dfs_monitor["120_gd_60_bess_df"])

storages_soc_df = pd.DataFrame(dfs_storage)
#display(storages_soc_df)

display(df_energymeter)
print(dfs_voltage["voltage_trf_5286812a_120_gd_60_bess_df"].columns)

,Caso,Perdas (kWh),Energia (kWh),Energia injetada (kWh)
0,Caso Base,6277.866346,124421.081474,0.0
1,Caso com 60 GDs,5348.542958,78896.409729,-26567.828373
2,Caso com 60 GDs e 30 Baterias,4216.628322,82135.274434,-11292.144691
3,Caso com 120 GDs,5882.456263,49444.317416,-43688.849394
4,Caso com 120 GDs e 60 Baterias,3434.882499,54543.235542,-17469.896441


Index(['Hour', 'sec', ' V1', ' VAngle1', ' V2', ' VAngle2', ' V3', ' VAngle3',
       ' V4', ' VAngle4', ' I1', ' IAngle1', ' I2', ' IAngle2', ' I3',
       ' IAngle3', ' I4', ' IAngle4'],
      dtype='str')
